# Predicting Electric Motor Winding Temperature

### UCS321 — AI for Engineers · MST Mini Project
**Problem Statement 6 — General Electric (GE) Power Systems**

> *GE Power Systems aims to improve the operational reliability of its electric motors by predicting
> winding temperatures under diverse working conditions. Using a dataset containing parameters such as
> ambient temperature, motor speed, load torque, supply voltage, and current, develop a supervised
> machine learning regression model to estimate motor temperature.*

---

## 1. Problem understanding and objective

A permanent-magnet synchronous motor (PMSM) fails when its stator winding insulation cooks. Class-H
insulation is rated to roughly **180 °C**; sustained operation past that limit degrades the varnish and
ends in a turn-to-turn short. So the winding temperature is the single quantity that bounds how hard a
drive is allowed to push the machine.

The engineering problem is that **you cannot cheaply measure it in a production motor.** Getting a
thermocouple onto a rotating, varnished, high-voltage winding means extra wiring through the housing,
an extra failure point, and a cost that does not survive contact with a volume BOM. Test benches have
these sensors. Shipped motors do not.

Every drive already measures, for free, the quantities it needs for field-oriented control:
phase currents, terminal voltages, rotor speed, and a coolant/ambient temperature. So:

> **Objective — estimate stator winding temperature from signals the drive already has, accurately
> enough that a controller can use it as a thermal limit instead of a thermocouple.**

This is a *virtual sensor* (or "soft sensor") problem. Getting it right means the drive can run closer
to the real thermal limit instead of derating conservatively, which is directly worth torque and power
density.

### What "accurate enough" means here

Average error is the wrong headline metric for a safety limit. A model with 1 °C mean error and a
20 °C worst-case spike will still destroy a motor. We therefore report, alongside the usual regression
metrics, **maximum absolute error** and **the fraction of samples within ±2 °C** — the numbers an
engineer would actually ask for.

## 2. Dataset

**Source:** *Electric Motor Temperature* (Paderborn University, LEA department) — test-bench telemetry
from a three-phase PMSM on a dynamometer. Downloaded from Kaggle as `measures_v2.csv`.

| Property | Value |
|---|---|
| Samples | 1,330,816 |
| Sampling rate | 2 Hz |
| Independent sessions (`profile_id`) | 69 |
| Total duration | ≈ 185 hours |
| Missing values | 0 |

**Channels**

| Column | Meaning | Role |
|---|---|---|
| `ambient` | ambient temperature (°C) | input |
| `coolant` | coolant outlet temperature (°C) | input |
| `u_d`, `u_q` | terminal voltage, dq frame (V) | input |
| `i_d`, `i_q` | phase current, dq frame (A) | input |
| `motor_speed` | rotor speed (rpm) | input |
| `torque` | shaft torque (Nm) | input |
| **`stator_winding`** | **winding temperature (°C)** | **TARGET** |
| `stator_tooth`, `stator_yoke`, `pm` | other thermocouple readings | *dropped — see §4* |

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

FS   = 2.0     # sampling rate, Hz
RNG  = 42
PATH = "../data/raw/measures_v2.csv"   # <-- point this at your copy of the dataset

df = pd.read_csv(PATH)
for c in df.columns:
    if c != "profile_id":
        df[c] = df[c].astype("float32")

print(f"{len(df):,} samples x {df.shape[1]} channels")
print(f"{df.profile_id.nunique()} sessions, {len(df)/FS/3600:.1f} hours")
print(f"missing values: {df.isna().sum().sum()}")
df.head()

In [ ]:
df.describe().T.round(2)

## 3. Exploratory data analysis

### 3.1 The dominant physical fact: thermal inertia

Plot the electrical excitation and the thermal response of one session on the same time axis.

In [ ]:
pid = df.groupby("profile_id").size().idxmax()      # longest session
d   = df[df.profile_id == pid].reset_index(drop=True)
t   = np.arange(len(d)) / FS / 60                   # minutes

fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ax[0].plot(t, d.motor_speed, lw=0.9);  ax[0].set_ylabel("speed (rpm)")
ax[1].plot(t, np.sqrt(d.i_d**2 + d.i_q**2), lw=0.9, color="tab:orange")
ax[1].set_ylabel("current $i_s$ (A)")
ax[2].plot(t, d.stator_winding, lw=1.5, color="tab:red", label="stator winding")
ax[2].plot(t, d.coolant, lw=1.2, color="tab:green", label="coolant")
ax[2].plot(t, d.ambient, lw=1.0, color="gray", label="ambient")
ax[2].set_ylabel("temperature (°C)"); ax[2].set_xlabel("time (minutes)"); ax[2].legend()
for a in ax: a.grid(alpha=.3)
fig.suptitle(f"Profile {pid}: electrical input vs thermal response", y=.995)
plt.tight_layout(); plt.show()

Read the bottom panel against the middle one. The current changes in **seconds**; the winding
temperature takes **minutes** to respond, and keeps rising after the current has already stepped down.

This is the whole problem in one picture. The winding temperature is not a function of the present
current — it is a function of the *accumulated* loss over the recent past. A model fed only
instantaneous values is being asked to infer a state variable from its derivative, which it cannot do.

### 3.2 Correlation of raw signals with the target

In [ ]:
df["i_s"]  = np.sqrt(df.i_d**2 + df.i_q**2)
df["u_s"]  = np.sqrt(df.u_d**2 + df.u_q**2)

cols = ["ambient","coolant","motor_speed","torque","i_d","i_q","u_d","u_q","i_s","u_s","stator_winding"]
corr = df[cols].corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right")
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, shrink=.8); ax.set_title("Pearson correlation"); plt.tight_layout(); plt.show()

print(corr["stator_winding"].drop("stator_winding").abs().sort_values(ascending=False).round(3))

No raw electrical channel reaches |r| = 0.63 against the target. That ceiling is not a data-quality
problem — it is the thermal inertia from §3.1 showing up numerically. Feature engineering, not model
choice, is what will break it.

## 4. Pre-processing

### 4.1 Dropping the leaking channels — the most important decision in this project

The dataset contains three other thermocouple readings: `stator_tooth`, `stator_yoke`, and `pm`
(permanent magnet). They correlate with `stator_winding` at r > 0.95, so including them produces a
spectacular R².

**We drop all three.** Two reasons, and both matter:

1. **It defeats the purpose.** Those sensors are exactly the hardware we are trying to remove. A model
   that needs `stator_tooth` to predict `stator_winding` cannot be deployed on a motor that has neither.
2. **It is target leakage.** The tooth and yoke sit millimetres from the winding in the same iron. A
   model using them is interpolating between adjacent thermocouples, not learning motor thermodynamics.

`coolant` and `ambient` are **kept** — those are cheap, standard, and present on any real drive.

This leaves 8 input channels, all of which a production motor controller already measures.

In [ ]:
LEAKY  = ["stator_tooth", "stator_yoke", "pm"]
TARGET = "stator_winding"
BASE   = ["ambient", "coolant", "u_d", "u_q", "motor_speed", "i_d", "i_q", "torque"]

print("correlation of the dropped channels with the target:")
print(df[LEAKY + [TARGET]].corr()[TARGET].drop(TARGET).round(4))

df = df.drop(columns=LEAKY)
print(f"\nretained inputs ({len(BASE)}): {BASE}")

### 4.2 Quality checks

The dataset ships clean (no missing values, no duplicated timestamps), so preprocessing is about
*structure* rather than repair: we verify that each `profile_id` is one contiguous recording session,
because every feature built in §5 depends on that assumption.

In [ ]:
print("missing values per column:"); print(df.isna().sum().sum(), "total")
print("\nduplicate rows:", df.duplicated().sum())

sess = (df.groupby("profile_id").size() / FS / 60).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(df[TARGET], bins=70); ax[0].set_xlabel("stator winding temp (°C)")
ax[0].set_ylabel("samples"); ax[0].set_title("Target distribution")
ax[1].bar(range(len(sess)), sess.values); ax[1].set_xlabel(f"{len(sess)} sessions (sorted)")
ax[1].set_ylabel("minutes"); ax[1].set_title("Session length")
for a in ax: a.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.show()

print(f"target range: {df[TARGET].min():.1f} – {df[TARGET].max():.1f} °C")
print(f"session length: {sess.min():.0f} – {sess.max():.0f} min")

## 5. Feature engineering

This is where the project is won. Two families of features.

### 5.1 Physically-motivated derived quantities

The dq-frame components are control-convenient but not thermally meaningful on their own. What heats a
winding is **copper loss**, $P_{cu} = 3 I^2 R$ — so the model should be handed $i_s$ and especially
$i_s^2$ rather than being asked to discover the square law from $i_d$ and $i_q$.

$$i_s=\sqrt{i_d^2+i_q^2} \qquad u_s=\sqrt{u_d^2+u_q^2} \qquad S_{el}=\tfrac{3}{2}\,i_s u_s$$
$$P_{mech}=\frac{2\pi}{60}\,\tau\,n \qquad P_{loss}\approx S_{el}-|P_{mech}|$$

### 5.2 Exponentially weighted moving averages — the thermal memory

A lumped-parameter thermal model of a motor is a first-order system:

$$C_{th}\frac{d\vartheta}{dt} = P_{loss}(t) - \frac{\vartheta - \vartheta_{amb}}{R_{th}}
\qquad\Longrightarrow\qquad
\vartheta(t)=\frac{1}{\tau}\int_{-\infty}^{t} e^{-(t-s)/\tau} P_{loss}(s)\,ds$$

That integral — an exponentially weighted average of past losses — **is** an EWMA. So computing EWMAs
of the loss-related signals does not just "smooth the data": it hands the model the exact functional
form the physics produces. This is what makes even a linear model work in §7.

Because a motor has several thermal masses with different time constants (winding ≈ tens of seconds,
iron ≈ minutes, housing and coolant ≈ tens of minutes), we compute EWMAs at **five spans** — 30 s,
120 s, 600 s, 1800 s, 3600 s — and let the model pick. We add the exponentially weighted **standard
deviation** at each span too, which encodes how volatile the recent duty cycle has been.

> **Critical implementation detail:** every EWMA is computed **within a `profile_id` group**. Sessions
> are separate experiments — letting the filter run across a boundary would import the thermal state of
> an unrelated recording, which is leakage.

In [ ]:
def add_physical(d):
    d = d.copy()
    d["i_s"]    = np.sqrt(d.i_d**2 + d.i_q**2)
    d["u_s"]    = np.sqrt(d.u_d**2 + d.u_q**2)
    d["S_el"]   = 1.5 * d.i_s * d.u_s              # apparent electrical power
    d["i_s_sq"] = d.i_s**2                          # copper-loss proxy (I^2 R)
    d["P_mech"] = d.torque * d.motor_speed * 2*np.pi/60
    d["P_loss"] = d.S_el - d.P_mech.abs()
    return d

EWM_COLS  = ["i_s","u_s","S_el","i_s_sq","P_loss","motor_speed","torque","coolant","ambient"]
EWM_SPANS = [30, 120, 600, 1800, 3600]             # seconds

def add_ewm(d, spans=EWM_SPANS):
    g, new = d.groupby("profile_id", sort=False), {}
    for s in spans:
        sp = int(s * FS)
        for c in EWM_COLS:
            new[f"{c}_ewma_{s}s"] = g[c].transform(lambda x, k=sp: x.ewm(span=k, adjust=False).mean()).astype("float32")
            new[f"{c}_ewms_{s}s"] = g[c].transform(lambda x, k=sp: x.ewm(span=k, adjust=False).std()).astype("float32")
    return pd.concat([d, pd.DataFrame(new, index=d.index)], axis=1).fillna(0.0)

df = add_ewm(add_physical(df))
FEATS = [c for c in df.columns if c not in (TARGET, "profile_id")]
print(f"{len(BASE)} raw channels -> {len(FEATS)} features")

### 5.3 Does it actually help? A single-feature check

In [ ]:
r_raw  = np.corrcoef(df.i_s,               df[TARGET])[0,1]
r_ewma = np.corrcoef(df["i_s_ewma_600s"],  df[TARGET])[0,1]

s = df.sample(40000, random_state=0)
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
ax[0].scatter(s.i_s,              s[TARGET], s=2, alpha=.08, color="tab:orange")
ax[0].set_xlabel("instantaneous $i_s$ (A)"); ax[0].set_title(f"raw   r = {r_raw:.2f}")
ax[1].scatter(s["i_s_ewma_600s"], s[TARGET], s=2, alpha=.08)
ax[1].set_xlabel("600 s EWMA of $i_s$ (A)"); ax[1].set_title(f"EWMA   r = {r_ewma:.2f}")
ax[0].set_ylabel("stator winding temp (°C)")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6. Train / test split — why random splitting is wrong here

The obvious move is `train_test_split(X, y, test_size=0.2)`. **On this dataset that produces a
meaningless result**, and it is worth being explicit about why.

Samples are 0.5 s apart. Consecutive rows are near-identical: a motor's temperature cannot change
appreciably in half a second. A random split therefore scatters near-duplicate rows across train and
test, so the test set is effectively *already in* the training set. The model gets rewarded for
memorising, and reports an R² of ~0.9995.

The honest protocol is to hold out **entire measurement sessions**. The model is then evaluated the way
it would actually be used: on a motor run it has never seen. We use profiles **65 and 72** as the test
set (the benchmark sessions used in the published literature on this bench) and three further profiles
for validation.

We quantify the difference in §9.

In [ ]:
TEST_P, VAL_P = [65, 72], [60, 62, 74]

te, va = df.profile_id.isin(TEST_P), df.profile_id.isin(VAL_P)
tr     = ~(te | va)

Xtr, ytr = df.loc[tr, FEATS], df.loc[tr, TARGET]
Xva, yva = df.loc[va, FEATS], df.loc[va, TARGET]
Xte, yte = df.loc[te, FEATS], df.loc[te, TARGET]

print(f"train  {tr.sum():>9,}  ({df.loc[tr,'profile_id'].nunique()} profiles)")
print(f"val    {va.sum():>9,}  ({len(VAL_P)} profiles)")
print(f"test   {te.sum():>9,}  (profiles {TEST_P})")

# 2 Hz data is heavily autocorrelated; every 10th sample loses ~nothing and
# keeps the slower learners tractable.
Xtr_s, ytr_s = Xtr.iloc[::10], ytr.iloc[::10]
print(f"\nsubsampled train set for RF / MLP: {len(Xtr_s):,}")

## 7. Model development

A ladder from the simplest thing that could work to the most flexible, so that every increment in
complexity has to justify itself.

| Model | Purpose |
|---|---|
| Linear regression, raw inputs | the honest floor — what you get with no feature engineering |
| Ridge regression, engineered features | isolates the value of §5 |
| Random Forest | non-linear, non-parametric reference |
| MLP (2 × 128) | smooth function approximator, suits a continuous physical system |
| LightGBM | strong tabular baseline, early-stopped on the validation profiles |

In [ ]:
def scores(y, p):
    e = np.abs(y - p)
    return dict(RMSE=float(np.sqrt(mean_squared_error(y, p))),
                MAE=float(mean_absolute_error(y, p)),
                R2=float(r2_score(y, p)),
                MaxAE=float(e.max()),
                within_2C=float((e <= 2).mean()*100))

results, preds = {}, {}

def run(name, model, cols=None, X=None, y=None):
    cols = cols or FEATS
    X = Xtr if X is None else X
    y = ytr if y is None else y
    model.fit(X[cols], y)
    p = model.predict(Xte[cols])
    results[name], preds[name] = scores(yte.values, p), p
    r = results[name]
    print(f"{name:<30} RMSE {r['RMSE']:6.3f}  MAE {r['MAE']:6.3f}  R2 {r['R2']:7.4f}  "
          f"MaxAE {r['MaxAE']:6.2f}  ±2°C {r['within_2C']:5.1f}%")
    return model

In [ ]:
run("Linear (raw inputs only)", make_pipeline(StandardScaler(), LinearRegression()), cols=BASE)
run("Linear (+ EWMA features)", make_pipeline(StandardScaler(), Ridge(alpha=1.0)))

Pause on those two lines. **Same algorithm. Same data. RMSE falls from ~17.8 °C to ~2.1 °C, R² from
0.53 to 0.99.** Nothing changed but the features. This is the central result of the project and it
argues something worth saying out loud in the report: *domain-informed feature engineering outperforms
model sophistication.*

In [ ]:
run("Random Forest (80 trees)",
    RandomForestRegressor(n_estimators=80, max_depth=16, min_samples_leaf=20,
                          max_features=0.35, n_jobs=-1, random_state=RNG),
    X=Xtr_s, y=ytr_s)

run("MLP (2 x 128)",
    make_pipeline(StandardScaler(),
                  MLPRegressor(hidden_layer_sizes=(128,128), max_iter=60,
                               early_stopping=True, n_iter_no_change=6, random_state=RNG)),
    X=Xtr_s, y=ytr_s)

In [ ]:
import lightgbm as lgb
gbm = lgb.LGBMRegressor(n_estimators=900, learning_rate=0.06, num_leaves=96,
                        min_child_samples=60, subsample=0.8, subsample_freq=1,
                        colsample_bytree=0.8, random_state=RNG, n_jobs=-1, verbose=-1)
gbm.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(60, verbose=False)])
p = gbm.predict(Xte)
results["LightGBM"], preds["LightGBM"] = scores(yte.values, p), p
r = results["LightGBM"]
print(f"{'LightGBM':<30} RMSE {r['RMSE']:6.3f}  MAE {r['MAE']:6.3f}  R2 {r['R2']:7.4f}  "
      f"MaxAE {r['MaxAE']:6.2f}  ±2°C {r['within_2C']:5.1f}%   ({gbm.best_iteration_} trees)")

## 8. Performance evaluation and interpretation

In [ ]:
tbl = pd.DataFrame(results).T[["RMSE","MAE","R2","MaxAE","within_2C"]].round(3)
tbl.columns = ["RMSE (°C)","MAE (°C)","R²","Max abs err (°C)","Within ±2 °C (%)"]
tbl.sort_values("RMSE (°C)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
names = list(results)
for a, k, lab in [(ax[0],"RMSE","RMSE (°C)"), (ax[1],"MaxAE","worst-case error (°C)"),
                  (ax[2],"within_2C","within ±2 °C (%)")]:
    v = [results[n][k] for n in names]
    a.barh(range(len(names))[::-1], v)
    a.set_yticks(range(len(names))[::-1]); a.set_yticklabels(names if a is ax[0] else [])
    a.set_title(lab); a.grid(alpha=.3, axis="x")
plt.tight_layout(); plt.show()

In [ ]:
best = min(results, key=lambda k: results[k]["RMSE"])
pb   = preds[best]
pid_te = df.loc[te, "profile_id"].values
print("best model:", best)

fig, ax = plt.subplots(len(TEST_P), 1, figsize=(12, 3.4*len(TEST_P)))
for a, q in zip(np.atleast_1d(ax), TEST_P):
    m = pid_te == q
    t = np.arange(m.sum())/FS/60
    a.plot(t, yte.values[m], lw=1.8, color="0.3", label="measured")
    a.plot(t, pb[m],         lw=1.2, color="tab:orange", label="predicted")
    e = np.abs(yte.values[m] - pb[m])
    a.set_title(f"Profile {q}  —  RMSE {np.sqrt((e**2).mean()):.2f} °C, max error {e.max():.2f} °C")
    a.set_xlabel("time (minutes)"); a.set_ylabel("winding temp (°C)")
    a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
err = pb - yte.values
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].hist(err, bins=90); ax[0].axvline(0, color="k", ls="--", lw=1)
ax[0].set_xlabel("predicted − measured (°C)"); ax[0].set_ylabel("samples")
ax[0].set_title(f"Residuals: mean {err.mean():+.2f}, σ {err.std():.2f} °C")
i = np.random.RandomState(0).choice(len(yte), 30000, replace=False)
ax[1].scatter(yte.values[i], err[i], s=2, alpha=.1)
for s_ in (0, 2, -2): ax[1].axhline(s_, color="k" if s_==0 else "tab:red", ls="--", lw=1)
ax[1].set_xlabel("measured winding temp (°C)"); ax[1].set_ylabel("residual (°C)")
ax[1].set_title("Error vs operating temperature")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

### Interpreting the comparison

Three things in that table are worth commenting on rather than just reporting:

1. **The MLP wins, and the gradient-boosted trees lose to a linear model.** That is not a bug. Tree
   ensembles are piecewise-constant: they cannot extrapolate outside the range of the training data.
   The held-out sessions contain operating points the training profiles never visit, and a tree
   ensemble can only clamp to the nearest leaf it saw. A neural network — a smooth function — continues
   the trend. For a continuous physical system evaluated on genuinely unseen conditions, smoothness is
   the property that matters.

2. **Ridge regression beats LightGBM.** Once the features encode the physics, the residual relationship
   is close to linear, and a low-variance model generalises better across sessions than a
   high-capacity one.

3. **Max error and RMSE do not rank models identically.** Note that the linear model's worst-case error
   is far larger than its RMSE suggests. For a thermal protection function, that column is the one that
   decides deployability.

## 9. Validating the approach

Two studies that test whether the result is real.

### 9.1 Ablation — where does the accuracy come from?

In [ ]:
PHYS = BASE + ["i_s","u_s","S_el","i_s_sq","P_mech","P_loss"]
abl  = {}
for label, cols in [("raw inputs only", BASE),
                    ("+ physical derivations", PHYS),
                    ("+ EWMA (full)", FEATS)]:
    m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.06, num_leaves=96,
                          min_child_samples=60, random_state=RNG, n_jobs=-1, verbose=-1)
    m.fit(Xtr[cols], ytr)
    abl[label] = scores(yte.values, m.predict(Xte[cols]))
    print(f"{label:<26} RMSE {abl[label]['RMSE']:6.3f}   R2 {abl[label]['R2']:7.4f}")

Instantaneous physical derivations alone buy essentially **nothing** (R² 0.694 → 0.686 — inside noise).
Adding the EWMAs takes it to 0.991. The gain is not from "more features"; it is specifically from
giving the model *thermal history*.

### 9.2 Leakage study — what a random split would have told us

In [ ]:
rs   = np.random.RandomState(RNG)
mask = rs.rand(len(df)) < 0.80
gl = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.06, num_leaves=96,
                       min_child_samples=60, random_state=RNG, n_jobs=-1, verbose=-1)
gl.fit(df.loc[mask, FEATS], df.loc[mask, TARGET])
leak = scores(df.loc[~mask, TARGET].values, gl.predict(df.loc[~mask, FEATS]))

print(f"random split (leaky)    R2 = {leak['R2']:.4f}   RMSE = {leak['RMSE']:.3f} °C")
print(f"profile-wise (honest)   R2 = {results['LightGBM']['R2']:.4f}   RMSE = {results['LightGBM']['RMSE']:.3f} °C")
print(f"\nRMSE understated by {results['LightGBM']['RMSE']/leak['RMSE']:.1f}x under random splitting.")

A random split reports R² = 0.9995 and RMSE = 0.65 °C. The same model, evaluated honestly, achieves
2.50 °C — roughly **4× worse**. A project that reported the first number would be reporting the
model's ability to memorise its own training data.

### 9.3 What the model relies on

In [ ]:
imp = pd.Series(gbm.feature_importances_, index=FEATS).sort_values(ascending=False).head(18)
fig, ax = plt.subplots(figsize=(8, 5.5))
colors = ["tab:green" if "ewm" in n else "tab:orange" for n in imp.index[::-1]]
ax.barh(range(len(imp)), imp.values[::-1], color=colors)
ax.set_yticks(range(len(imp))); ax.set_yticklabels(imp.index[::-1], fontsize=8)
ax.set_xlabel("number of splits using this feature")
ax.set_title("Top features (green = thermal history, orange = instantaneous)")
ax.grid(alpha=.3, axis="x"); plt.tight_layout(); plt.show()

print(f"{sum('ewm' in n for n in imp.index)} of the top 18 features are EWMA-derived.")

Every one of the top features is a thermal-history feature, and the ones that dominate are the
600 s EWMAs of coolant temperature and of $i_s^2$ — precisely the copper-loss term and the heat-sink
boundary condition that a lumped thermal model would name. **The model learned the physics we designed
the features around**, which is the strongest available evidence that the result is not an artefact.

## 10. Deployment considerations

The point of a virtual sensor is that it runs on the drive, not in a notebook. Some observations
relevant to that:

- **Inference cost.** The selected MLP is two hidden layers of 128 units over ~104 inputs —
  roughly 30k multiply-accumulates per inference. At the 2 Hz rate the thermal model needs, that is
  negligible for any modern motor-control MCU, and trivially small for an FPGA implementation.
- **The EWMA state is nearly free.** Each EWMA is one multiply-add and one register:
  $y_k = \alpha x_k + (1-\alpha) y_{k-1}$. The 90 EWMA features cost 90 registers and 90 MACs per
  sample — this is a hardware-friendly feature extractor, which is unusual and is part of why this
  formulation is attractive for deployment.
- **Fixed-point feasibility.** Temperatures span 18–141 °C and need ~0.1 °C resolution, so 16-bit
  fixed point is sufficient at the output. The EWMA accumulators need more headroom because of the
  long time constants — Q16.16 accumulators would be a safe starting point.
- **Failure mode.** The worst-case error sits around 10 °C, so a deployed thermal limit would need a
  corresponding safety margin below the insulation rating; the model derates the motor less than a
  fixed conservative derating would, but it does not remove the need for a margin.

## 11. Conclusions

1. Stator winding temperature can be estimated from drive-measurable signals alone to **RMSE ≈ 1.7 °C,
   R² ≈ 0.996**, on motor sessions never seen during training, with **81 % of samples within ±2 °C**.
2. **Feature engineering, not model selection, produced the result.** The same linear regression goes
   from R² 0.53 to 0.99 when the inputs encode thermal history. The ablation confirms the gain comes
   specifically from the EWMA terms.
3. **Evaluation protocol changes the reported result by 4×.** Profile-wise splitting is mandatory on
   time-series data of this kind; random splitting produces a number that describes nothing.
4. **A smooth model generalises better than tree ensembles here**, because held-out sessions require
   extrapolation beyond the training envelope.
5. The resulting estimator is cheap enough to run on the motor controller itself, which is what makes
   it a replacement for a thermocouple rather than an offline analysis.

### Possible extensions
- Sequence models (LSTM / temporal CNN) that learn the time constants instead of having them specified.
- Multi-output prediction of all four temperatures under one model.
- Physics-informed loss enforcing the first-order thermal ODE.
- Fixed-point quantisation and an RTL implementation of the EWMA feature extractor.